# Trend Tracker — Notebook 03: Insights Generation

Runs the full analytical pipeline on the enriched corpus, from TF-IDF and NMF
topic discovery through LLM synthesis, verification, packaging, and DOCX report
generation.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Step | What | Key outputs |
|------|------|-------------|
| 1 | Build TF-IDF matrices | in-memory for Steps 2–4 |
| 2 | Quality checkpoint 2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `analysis/project_topic_bridge.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |
| 6 | Synthesis | `analysis/llm_synthesis_*.txt` |
| 7 | Structured insight extraction | `insights/insights_candidates.json` |
| 8 | Topic verification | (updates insights in-memory) |
| 9 | Evidence tables | `insights/insights_flat.csv`, `insights/insight_topic_support_candidates.csv` |
| 10 | Packaging, dedupe & topline | (updates curated insights in-memory) |
| 11 | Final outputs & manifests | `reports/trend_tracker_report.docx`, `insights/insights_structured.json` |

---
## Setup and Configuration

In [1]:
import sys
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import pandas as pd
import numpy as np

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

import importlib.util as _ilu, sys as _sys
_u = next(Path(".").glob("utils*.py"))
if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u); _m = _ilu.module_from_spec(_s); _s.loader.exec_module(_m); _sys.modules["utils"] = _m
    
from utils import (
    load_cfg,
    write_json,
    artifact_meta,
    build_run_output_path,
    get_run_date,
    canonicalize_filter_spec,
    get_filter_fields_key,
    get_run_id,
    apply_filters,
    ensure_warning_file,
    append_warning,
    get_llm_client,
    start_stage_manifest,
    finalize_stage_manifest,
    build_pipeline_manifest,
    tokens_to_str,
    make_vec,
    add_bin,
    group_key,
    build_project_topic_bridge,
    quality_report,
    cat_tfidf_slice,
    nmf_one,
    build_input,
    _label_with_retry,
    _norm_group_value,
    _safe_topic_id,
    build_topic_lines,
    _call_with_retry,
    synthesize_one_group,
    strip_json_fences,
    normalize_insight,
    build_bridge_lookup,
    build_label_index,
    build_verified_insight_tables,
    apply_deterministic_packaging,
    dedupe_packaged_insights,
    assign_topline_sections_simple,
    build_structured_from_curated,
    build_looker_project_url,
    build_packaged_report_docx,
    project_insight_for_saved_candidates,
    _verify_insight_list,
    resolve_params_path,
)

CFG_PATH = resolve_params_path()
CFG      = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)
ct       = CFG["tfidf"]
client   = get_llm_client()

# Load the enriched corpus produced by NB02. To use the unenriched baseline
# instead, change the path to 05_filtered.parquet.
raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/06_enriched.parquet")

# Stopwords for the quality gate at Step 2. Loaded from params.yaml so the
# list can be extended without touching notebook code.
STOPWORDS = CFG["quality"]["stopword_violation_list"]

print(f"Loaded {len(raw_df):,} projects  |  {raw_df.shape[1]} columns")

Loaded 2,244,916 projects  |  172 columns


---
## Parameters

All runtime configuration is resolved here from `params.yaml`. Edit `params.yaml`
to change behaviour; do not hardcode values in downstream cells.

In [2]:
# ── New groupby fields ────────────────────────────────────────────────────────
# ── Funded/expired x metro, excluding live ────────────
expiration_dt = pd.to_datetime(raw_df.get("expiration_date"), errors="coerce")

status = pd.Series(
    np.select(
        [
            raw_df["funded_date"].notna(),
            expiration_dt.le(pd.Timestamp.today().normalize()),
        ],
        ["Funded", "Expired"],
        default=pd.NA,
    ),
    index=raw_df.index,
)

raw_df["funding_status_x_metro"] = (
    status.astype("string") + " | " +
    raw_df["metro_type_at_time_of_posting"].fillna("Unknown").astype("string")
).where(status.notna(), pd.NA)

# ── Derived groupby field: EFS x grade band ──────────────────────────────────
rural = raw_df["school_is_underserved_rural"].eq("Yes")
race  = raw_df["school_is_historically_underrepresented_race"].eq("Yes")
inc   = raw_df["school_is_low_income"].eq("Yes")

efs = pd.Series(
    np.select(
        [race & inc, rural, race, inc],
        ["Race+Inc", "Rural", "Race", "Inc"],
        default="NonEFS",
    ),
    index=raw_df.index,
)

raw_df["efs_x_grade_band"] = (
    efs.astype("string") + " | " +
    raw_df["grade_band"].fillna("Unknown").astype("string")
)

#raw_df["funding_status_x_metro"].value_counts(dropna=False)
#raw_df["efs_x_grade_band"].value_counts(dropna=False)

# ── Waterfall groupby fields from injected taxonomy tokens ────────────────────
# For each priority list, pick the first listed token present in the project's
# enriched token list; projects with none get "__NA__" so they can be excluded
# via exclude_groups or a filter in NB03.

import re

_PREFIX_RE = re.compile(r"^__(sensitive_context|subject|industry|context|framing)_")
_SUFFIX_RE = re.compile(r"_(context|framing)__$")

def short_name(tok):
    """Strip namespace prefix and trailing _context/_framing suffix for display."""
    return _SUFFIX_RE.sub("__", _PREFIX_RE.sub("__", tok))

def first_present(tokens, priority):
    s = set(tokens)
    match = next((t for t in priority if t in s), None)
    return short_name(match) if match else "__NA__"

WATERFALLS = {
    "specialty_subject": [
        "__subject_coding_computer_science__",
        "__subject_ai_literacy_foundations__",
        "__subject_restorative_practices_conflict_resolution__",
        "__subject_cte_credentials_certifications__",
        "__subject_multilingual_language_access__",
        "__subject_entrepreneurship_business_skills__",
        "__subject_ai_accessibility_assistive_use__",
        "__subject_science_inquiry_lab_learning__",
        "__subject_math_manipulatives_quantitative_learning__",
        "__subject_cooking_culinary_life_skills__",
        "__subject_financial_literacy_personal_finance__",
        "__subject_robotics_makerspace_3d_printing__",
        "__subject_climate_environmental_science_learning__",
        "__subject_nutrition_literacy_health_education__",
        "__subject_home_language_family_literacy__",
        "__subject_sel_curriculum_skill_building__",
        "__subject_assistive_adaptive_technology__",
        "__subject_food_systems_agriculture_learning__",
        "__subject_engineering_design_maker_learning__",
        "__subject_employability_durable_skills__",
        "__subject_english_language_development__",
        "__subject_phonics_decodable_reading_intervention__",
    ],
    "industry": [
        "__industry_agriculture_food_systems_careers__",
        "__industry_public_safety_civic_careers__",
        "__industry_green_jobs_sustainability_careers__",
        "__industry_transportation_logistics_aviation__",
        "__industry_robotics_engineering_careers__",
        "__industry_business_entrepreneurship_careers__",
        "__industry_health_careers__",
        "__industry_media_creative_industries__",
    ],
    "safety_justice": [
        "__sensitive_context_gang_violence_exposure_context__",
        "__sensitive_context_justice_involved_family_context__",
        "__sensitive_context_community_violence_safety_context__",
        "__context_alternative_school_justice_adjacent_context__",
        "__context_direct_attendance_absence_terms__",
        "__context_discipline_exclusion_absence__",
        "__context_mentoring_protective_support__",
    ],
    "future_framing": [
        "__framing_ai_literacy_responsible_use__",
        "__framing_ambitious_programmatic_ask__",
        "__framing_applied_production_real_world_execution__",
        "__framing_career_exploration_pathways__",
        "__framing_career_identity_exposure__",
        "__framing_coding_computational_thinking_foundation__",
        "__framing_future_preparation__",
        "__framing_opportunity_expansion__",
        "__framing_technical_pathway_skill_building__",
    ],
    "framing_context": [
        "__framing_belonging_identity_affirmation__",
        "__framing_catch_up_remediation__",
        "__framing_chronic_scarcity__",
        "__framing_disability_accommodation_access__",
        "__framing_event_news_disaster_anchoring__",
        "__framing_experiential_hands_on_learning__",
        "__framing_expressive_creative_production__",
        "__framing_home_extension_family_colearning__",
        "__framing_inclusion_accommodation_framing__",
        "__framing_rural_scarcity_dispersion__",
        "__framing_safety_protection_language__",
        "__framing_stigma_avoidance_social_confidence__",
        "__framing_student_voice_belonging_identity__",
    ],
}

for field, priority in WATERFALLS.items():
    raw_df[field] = raw_df["tokens"].apply(lambda ts, p=priority: first_present(ts, p))
    n = (raw_df[field] != "__NA__").sum()
    print(f"{field:28s}  {n:>6,} tagged  ({n/len(raw_df):.1%})")

specialty_subject             622,581 tagged  (27.7%)
industry                      243,983 tagged  (10.9%)
safety_justice                144,225 tagged  (6.4%)
future_framing                357,108 tagged  (15.9%)
framing_context               663,977 tagged  (29.6%)


In [3]:
# ── Analysis scope ────────────────────────────────────────────────────────────
GROUPBY_FIELD  = CFG["analysis"]["group_by"]
FILTER_LOGIC   = CFG["analysis"].get("filter_logic", "and")
FILTERS        = CFG["analysis"].get("filters", [])
EXCLUDE_GROUPS = CFG["analysis"]["exclude_groups"]
REVIEW_GROUP   = CFG["analysis"]["review_group"]

# Apply declarative filters first so all corpus-scope checks run on the
# right population.
df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters — check params.yaml.")

# ── v1 feature guards ─────────────────────────────────────────────────────────
# Binned/time-slice mode: topic identity and downstream traceability (bridge
# table, label index, source_topics) are keyed by group + topic_id only.
# Adding a bin dimension would cause silent topic-id collisions across bins
# within the same group. Guard here until topic identity is bin-aware.
if CFG["analysis"].get("bins", []):
    raise ValueError(
        "analysis.bins is non-empty: binned/time-slice mode is not supported "
        "in v1. Topic identity and source_topic traceability are not bin-aware. "
        "Set analysis.bins: [] to run without time bucketing."
    )

# review_group: restricting to a single group would bypass cross-group
# synthesis and require conditional output handling that is not wired in v1.
# Disable until the cross-group path is made optional.
if REVIEW_GROUP is not None:
    raise ValueError(
        f"analysis.review_group is set to {REVIEW_GROUP!r}: single-group review "
        "mode is not supported in v1. Set analysis.review_group: null to run "
        "against all groups."
    )

# ── Group scope: apply exclude_groups once, here, for the whole pipeline ──────
# Removing excluded groups from df means every downstream step (TF-IDF, NMF,
# labeling, synthesis) automatically excludes them. No per-step filtering needed.
if EXCLUDE_GROUPS:
    n_before = len(df)
    df = df[
        ~df[GROUPBY_FIELD].astype(str).isin([str(g) for g in EXCLUDE_GROUPS])
    ].reset_index(drop=True)
    if df.empty:
        raise ValueError(
            f"No rows remain after excluding groups {EXCLUDE_GROUPS} — "
            "check analysis.exclude_groups in params.yaml."
        )
    print(f"exclude_groups: {n_before - len(df):,} rows removed "
          f"({len(EXCLUDE_GROUPS)} group(s): {EXCLUDE_GROUPS})")

# ── Run identity ───────────────────────────────────────────────────────────────
# RUN_DATE and RUN_ID are assigned BEFORE OUT() is defined because OUT() closes
# over them. Reversing this order causes a NameError at the OUT() call site.
FILTER_SPEC       = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)
RUN_DATE          = get_run_date()
RUN_ID            = get_run_id(GROUPBY_FIELD, FILTER_SPEC)

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir, fname=fname,
        groupby_field=GROUPBY_FIELD, run_date=RUN_DATE, run_id=RUN_ID,
    )

# ── Analysis description and topic/slice settings ─────────────────────────────
GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
    GROUPBY_FIELD,
    f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data",
)
MIN_SHARED    = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE  = CFG["analysis"]["min_coverage"]
BASE_N_COMPONENTS = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N   = CFG["analysis"]["cat_tfidf_top_n"]

SLICE_RULES   = CFG["analysis"]["slice_rules"]
VERIFY_CFG    = CFG["analysis"]["verification"]
PACKAGING_CFG = CFG["analysis"]["packaging"]
DEDUPE_CFG    = CFG["analysis"]["dedupe"]

# group_project_counts is computed on the scoped df (post-filter, post-exclude)
# so small_slice_mode reflects the actual groups that will run.
group_project_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=["project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
)
median_group_projects = float(group_project_counts.median()) if not group_project_counts.empty else 0.0
SMALL_SLICE_MODE = median_group_projects < SLICE_RULES["small_slice_cutoff"]
SLICE_RULES      = {**SLICE_RULES, "small_slice_mode": SMALL_SLICE_MODE}

MIN_GROUP_PROJECTS_EFFECTIVE = max(
    CFG["analysis"]["min_group_projects"],
    SLICE_RULES["min_group_projects"],
)
CROSS_MIN_INSIGHTS     = SLICE_RULES["cross_min_insights"]
CROSS_MAX_INSIGHTS     = SLICE_RULES["cross_max_insights"]
PER_GROUP_MIN_INSIGHTS = SLICE_RULES["per_group_min_insights"]
PER_GROUP_MAX_INSIGHTS = SLICE_RULES["per_group_max_insights"]

# ── LLM settings ──────────────────────────────────────────────────────────────
N_REPRESENTATIVE          = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT       = CFG["llm"]["top_terms_in_prompt"]
SYNTHESIS_TOP_TERMS_COUNT = CFG["llm"]["synthesis_top_terms_count"]
# MAX_RETRIES is passed to all LLM call sites so retry behaviour is tunable
# from params.yaml without touching notebook code.
MAX_RETRIES = CFG["llm"]["max_retries"]

MODEL_LABELING  = CFG["models"]["labeling"]
MODEL_SYNTHESIS = CFG["models"]["synthesis"]
MODEL_VERIFY    = CFG["models"]["verify"]

MAX_WORKERS          = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"]["labeling_max_workers"]

# ── Output settings ────────────────────────────────────────────────────────────
LOOKER_BASE_URL     = CFG["output"]["looker_base_url"]
LOOKER_FILTER_FIELD = CFG["output"]["looker_filter_field"]
LOOKER_FIELDS       = CFG["output"]["looker_fields"]
LOOKER_LIMIT        = int(CFG["output"].get("looker_limit", 500))
LOOKER_ID_LIMIT     = int(CFG["output"].get("looker_id_limit", 100))

# CSV_MAX_IDS_PER_INSIGHT: ranked project IDs stored per insight row in
# insights_flat.csv. Passed to build_verified_insight_tables().
CSV_MAX_IDS_PER_INSIGHT = int(CFG["output"]["csv_max_ids_per_insight"])

# MAIN_MIN_VERIFICATION_RATIO: eligibility gate for main-section topline
# placement. Passed to assign_topline_sections_simple().
MAIN_MIN_VERIFICATION_RATIO = PACKAGING_CFG["main_min_verification_ratio"]

# bins are guarded out above; group_cols is always [GROUPBY_FIELD] in v1.
group_cols = [GROUPBY_FIELD]

# ── Warnings, filter provenance, stage manifest ────────────────────────────────
WARNINGS_PATH       = OUT("metadata", "warnings_03.jsonl")
FILTER_SPEC_PATH    = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
COPIED_CONFIG_PATH  = OUT("metadata", CFG_PATH.name)

ensure_warning_file(WARNINGS_PATH)
write_json(FILTER_SPEC_PATH, {
    "schema_version": "v1", "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD, "filter_fields_key": FILTER_FIELDS_KEY,
    "filter_logic": FILTER_LOGIC, "filters": FILTERS,
})
write_json(FILTER_SUMMARY_PATH, {
    "schema_version": "v1", "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD, **filter_summary,
})
shutil.copy2(CFG_PATH, COPIED_CONFIG_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation_v1.ipynb",
    config_path=MANIFEST_CFG_PATH,
    
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

print(f"RUN_ID            = {RUN_ID}")
print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY = {FILTER_FIELDS_KEY}")
print(f"Filtered rows     = {len(df):,} / {len(raw_df):,}")
print(f"small_slice_mode  = {SMALL_SLICE_MODE}  (median group = {median_group_projects:.1f})")
print(f"Output root       = OUTPUTS/runs/{GROUPBY_FIELD}/{RUN_DATE}/{RUN_ID}/")

exclude_groups: 656,750 rows removed (1 group(s): ['__NA__'])
RUN_ID            = 20260514_112716_industry_0c79fb23
GROUPBY_FIELD     = 'industry'
FILTER_FIELDS_KEY = expiration_date
Filtered rows     = 65,376 / 2,244,916
small_slice_mode  = False  (median group = 7014.5)
Output root       = OUTPUTS/runs/industry/2026-05-14/20260514_112716_industry_0c79fb23/


---
## Step 1 — Build TF-IDF Matrices

Fits one TF-IDF vectorizer per n-gram range on the full filtered corpus and
keeps the resulting sparse matrices in memory. All downstream steps reuse these
objects rather than refitting.

Trigrams are built when `ngrams.max_n >= 3` in `params.yaml`.

In [4]:
docs = df["tokens"].apply(tokens_to_str).tolist()

# Build one matrix per n-gram range. X_unigram_bigram is the primary matrix
# for category TF-IDF and NMF; the others are retained for debugging and
# future analytical passes.
specs = {
    "X_unigram":        (1, 1),
    "X_bigram":         (2, 2),
    "X_unigram_bigram": (1, 2),
}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name] = vec
    sz, nnz = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz / (sz[0] * sz[1]):.3f}")

# Spot-check first features to catch bad filtering or token drift.
for name, vec in vecs.items():
    print(f"  {name:20s} first feats → {vec.get_feature_names_out()[:10].tolist()}")

  X_unigram           : shape=(65376, 6334)  sparsity=0.991
  X_bigram            : shape=(65376, 67792)  sparsity=1.000
  X_unigram_bigram    : shape=(65376, 74126)  sparsity=0.999
  X_trigram           : shape=(65376, 13619)  sparsity=1.000
  X_unigram            first feats → ['aac', 'aba', 'abc', 'abcs', 'ability', 'able', 'above', 'abroad', 'absence', 'absent']
  X_bigram             first feats → ['aac communicate', 'aac device', 'aac technology', 'ability able', 'ability access', 'ability achieve', 'ability across', 'ability adapt', 'ability additionally', 'ability allow']
  X_unigram_bigram     first feats → ['aac', 'aac communicate', 'aac device', 'aac technology', 'aba', 'abc', 'abcs', 'ability', 'ability able', 'ability access']
  X_trigram            first feats → ['aac communicate world', 'ability build confidence', 'ability communicate effectively', 'ability control benefit', 'ability control want', 'ability critical thinke', 'ability engage activity', 'ability express ex

---
## Step 2 — Quality Checkpoint 2

Gate before LLM calls. Review before proceeding:

- **No stopword violations** — if terms from `quality.stopword_violation_list`
  appear in the top-200 vocab, re-run NB01 with tighter frequency thresholds.
- **Reasonable token distribution** — `p50` should be in the 20–60 range.
- **Matrix sparsity** — very high sparsity on the bigram matrix suggests
  `tfidf.min_df` may be too restrictive.

In [5]:
# Pass the configured stopword list so the gate reflects params.yaml rather
# than the HARD_STOPWORDS fallback in utils.py.
qr2 = quality_report(
    df, label="cp2",
    matrices=matrices,
    save_path=OUT("quality", "quality_cp2.json"),
    stopwords=STOPWORDS,
)


=======================================================  [cp2]
  Projects : 65,376
  Tok/proj : min=15  p50=57  max=100
  Vocab    : 7,421 unique tokens
  X_unigram           : shape=[65376, 6334]  sparsity=0.991
  X_bigram            : shape=[65376, 67792]  sparsity=1.000
  X_unigram_bigram    : shape=[65376, 74126]  sparsity=0.999
  X_trigram           : shape=[65376, 13619]  sparsity=1.000
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [6]:
# ── Category TF-IDF ────────────────────────────────────────────────────────
# Score each eligible group against the rest of the corpus using a shared matrix.

top_n = CAT_TFIDF_TOP_N
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

df_work = df.copy().reset_index(drop=True)
all_docs = df_work["tokens"].apply(tokens_to_str).tolist()

vec_cat = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full = vec_cat.fit_transform(all_docs)
feat = vec_cat.get_feature_names_out()
idf_vals = vec_cat.idf_

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd = group_key(keys, group_cols)
    top = cat_tfidf_slice(
        sub.index,
        df_index=df_work.index,
        X_full=X_full,
        feat=feat,
        idf_vals=idf_vals,
        top_n=top_n,
    )
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(
        f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml"
    )

cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)

print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

240 rows  |  8 groups


,industry,token,tf,idf,tfidf,prevalence,contrast,project_count
0,__agriculture_food_systems_careers__,farmer,0.037740,5.796357,0.218757,0.343312,0.343312,539
1,__agriculture_food_systems_careers__,agricultural,0.036609,5.807530,0.212605,0.339490,0.339490,533
2,__agriculture_food_systems_careers__,farm,0.030770,5.165282,0.158934,0.326115,0.318247,512
3,__agriculture_food_systems_careers__,farm animal,0.017670,6.745592,0.119193,0.132484,0.132484,208
4,__agriculture_food_systems_careers__,hydroponic garden,0.017205,6.721950,0.115649,0.135669,0.135669,213
5,__agriculture_food_systems_careers__,animal,0.025437,4.386273,0.111572,0.295541,0.268162,464
6,__agriculture_food_systems_careers__,hydroponic,0.019125,5.761776,0.110196,0.178344,0.173987,280
7,__agriculture_food_systems_careers__,garden,0.025061,4.367908,0.109462,0.305732,0.277961,480
8,__agriculture_food_systems_careers__,farmer market,0.015793,6.872990,0.108542,0.116561,0.116561,183
9,__agriculture_food_systems_careers__,agriculture,0.019231,5.429915,0.104422,0.210191,0.203170,330


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per group so the dominant vocabulary axis in one group
does not suppress signal in others. Topics are treated as evidence candidates
for LLM synthesis — not as stable theme definitions.

The NMF weight matrix `W` records how strongly each project loads on each topic.
Step 5 uses the top-weight projects per topic as representative snippets rather
than sampling randomly.

In [7]:
# ── Groupwise NMF topics ───────────────────────────────────────────────────
# Fit one NMF model per eligible group and keep both topic-level and project-level outputs.

cn = CFG["nmf"]
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

all_topics, all_weights = [], []
df_work = df.copy().reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    # Skip groups that are too small to support stable topic extraction.
    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W, nmf_meta = nmf_one(
            group_docs,
            ct_cfg=ct,
            cn_cfg=cn,
            base_n_components=BASE_N_COMPONENTS,
            slice_rules=SLICE_RULES,
        )
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    # None return means the slice was too thin; kept separate from exceptions for QA.
    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice was too thin",
            context={"group": group_value, **(nmf_meta or {})},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    # Persist the full W ranking so later cells can recover top projects per topic.
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append(
                {
                    **kd,
                    "topic_id": tid,
                    "project_id": pids[idx],
                    "weight": float(W[idx, tid]),
                    "rank": rank,
                }
            )

if not all_topics:
    raise RuntimeError(
        "No groups produced NMF topics — check n_components vs retained vocab, "
        "or lower min_group_projects"
    )

topics_df = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)

topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)

print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# Build the project-topic bridge once so downstream evidence collection can reuse it.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df,
    GROUPBY_FIELD,
    threshold,
)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(
    "Bridge: "
    f"{len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)

160 topics across 8 groups
Bridge: 52,890 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,industry
0,0,"[foster, sustainable, aim, enhance, deep, educ...","[0.8394078419737434, 0.5456739696780016, 0.534...",__agriculture_food_systems_careers__
1,1,"[sustainability cook, personal environmental, ...","[0.2383599281622342, 0.2383599281622342, 0.238...",__agriculture_food_systems_careers__
2,2,"[farm animal, farm, animal, play, toy, fun, ma...","[0.7159619092195529, 0.514959790721544, 0.5127...",__agriculture_food_systems_careers__
3,3,"[hydroponic, hydroponic garden, garden, plant,...","[0.7307718309656365, 0.6775950848478718, 0.529...",__agriculture_food_systems_careers__
4,4,"[dramatic, play, dramatic play, shop, market, ...","[0.6790046106621652, 0.5679147199233845, 0.497...",__agriculture_food_systems_careers__
5,5,"[agricultural, career, program, education, fut...","[0.4444957623444797, 0.323400313732517, 0.3157...",__agriculture_food_systems_careers__


In [8]:
# ── Cross-group universal themes ───────────────────────────────────────────
# Identify term bundles that recur across groups after topic extraction.

# Select the largest group as the cross-group reference point for any
# group-specific context needed in downstream steps.
REFERENCE_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

# theme_cats maps a shared term bundle to the set of groups where it recurs.
theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i + 1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append(
            {
                "theme": ", ".join(sorted(terms)[:5]),
                "n_groups": len(cats),
                "categories": sorted(cats),
            }
        )
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

,theme,n_groups,categories
0,"aim, enhance, environment, foster",7,"[__agriculture_food_systems_careers__, __busin..."
1,"able, class, some",6,"[__green_jobs_sustainability_careers__, __heal..."
2,"book, love, read",5,"[__agriculture_food_systems_careers__, __busin..."
3,"income, low, low income",5,"[__agriculture_food_systems_careers__, __green..."
4,"critical, critical thinke, problem, solve, thinke",5,"[__agriculture_food_systems_careers__, __green..."
5,"career, future, prepare",4,"[__agriculture_food_systems_careers__, __green..."
6,"career, future, prepare, program",4,"[__agriculture_food_systems_careers__, __busin..."
7,"book, library, read, reader",4,"[__agriculture_food_systems_careers__, __busin..."
8,"curiosity, explore, world",4,"[__agriculture_food_systems_careers__, __green..."
9,"critical, critical thinke, problem, problem so...",4,"[__agriculture_food_systems_careers__, __green..."


---
## Step 5 — LLM Topic Labeling

One API call per topic using compressed input — never raw essay text at scale.
Representative snippets are selected by NMF weight (highest-loading projects),
not randomly. Parallel execution via `ThreadPoolExecutor`.

Parse failures are stored with enough metadata to debug later; failed topics are
excluded from synthesis but do not halt the run.

In [9]:
# ── LLM topic labeling ─────────────────────────────────────────────────────
# Convert raw NMF topics into human-readable labels/descriptions with retry logic.

SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = rhetorical framing, tone, mechanism, or persuasion signal\n"
    "  __subject_[name]__ = subject/domain/content signal\n"
    "  __industry_[name]__ = workforce-development industry or skill-domain signal\n"
    "  __request_[name]__ = material/request-topology signal\n"
    "  __context_[name]__ = contextual school, community, attendance, safety, or access signal\n"
    "  __sensitive_context_[name]__ = direct sensitive-context signal; describe carefully and only when supported\n"
    "  __cat_[name]__ = legacy subject matter category token\n"
    "  __sub_[name]__ = legacy subcategory token\n"
    "These are meaningful signals. Use their meaning when clearly supported, but do not let them "
    "override stronger concrete evidence from repeated terms and snippets.\n"
    "Never output raw tokens such as __framing_*__, __subject_*__, __industry_*__, __request_*__, "
    "__context_*__, __sensitive_context_*__, __cat_*__, __sub_*__, or snake_case token names. "
    "Always translate such signals into plain English.\n\n"

    "Your job is to produce a stable canonical topic label and one dense grounded description.\n"
    "Do not optimize for cleverness, vividness, or donor-facing insight language. "
    "Optimize for specificity, repeatability, plain-English clarity, and preservation of useful evidence.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Prefer the most specific defensible mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Do not try to pack every nuance into proposed_label.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- Preserve concrete signals such as named programs, pedagogies, student populations, "
    "classroom routines, and rhetorical framing when clearly supported.\n\n"

    "Rules for description:\n"
    "- description must be exactly one sentence.\n"
    "- description must be plain English and evidence-led.\n"
    "- Use this order when supported by the evidence: "
    "(1) dominant mechanism or request pattern, "
    "(2) classroom function or use case, "
    "(3) population or setting, "
    "(4) framing signal in plain English.\n"
    "- Prefer concrete natural-language phrasing over abstract wording.\n"
    "- Do not write implications, recommendations, or donor-facing interpretation.\n"
    "- Do not use raw preprocessing tokens or token-like jargon in description.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for real edge cases.\n"
    "- If coherence_flag is mixed, briefly name the colliding subthemes.\n"
    "- If coherence_flag is redundant, briefly state the nature of the overlap.\n"
    "- If coherence_flag is unclear, briefly state why the topic is too scattered.\n"
    "- Do not restate the description in notes.\n\n"

    "coherence_flag definitions:\n"
    "  coherent   — one dominant theme, mechanism, population, or use case clearly leads, "
    "even if secondary signals are present.\n"
    "  mixed      — no single dominant theme clearly leads and two or more distinguishable "
    "subthemes are truly colliding.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label.\n\n"

    "Important tie-breaker:\n"
    "- If one theme is primary and another is secondary, mark the topic coherent, not mixed, "
    "and keep notes empty unless the secondary signal is important to preserve.\n"
    "- If two phrasings are both plausible, choose the more literal, reusable, and plain-English one."
)

PROMPT = (
    f"Group field: {GROUPBY_FIELD} — {GROUP_DESCRIPTION}\n"
    f"Group value: {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"

    "Instructions:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- description must be exactly one sentence in plain English.\n"
    "- Write description in this pattern when supported by the evidence: "
    "Requests for [mechanism/request] are used to [classroom function/use case] "
    "for [population or setting], often framed as [plain-English framing] when clearly present.\n"
    "- Prefer concrete mechanism and classroom function over broad category language.\n"
    "- Translate any framing/category/subcategory token signals into normal language; never copy raw token strings.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- Use coherence_flag='mixed' only when no single dominant theme clearly leads.\n"
    "- If one theme is primary and another is secondary, mark the topic coherent.\n"
    "- notes should usually be empty; use notes only for mixed, redundant, or unclear topics.\n"
    "- Do not write donor-facing insights, implications, or rhetorical flourishes.\n\n"

    "Output requirements:\n"
    "- proposed_label: short and stable\n"
    "- description: exactly one dense grounded sentence\n"
    '- coherence_flag: one of "coherent", "mixed", "redundant", "unclear"\n'
    "- notes: usually empty, unless needed for edge cases\n\n"

    f"Return a JSON object with exactly these keys:\n"
    f"{GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes"
)

# Build lightweight snippet text from the top projects attached to each topic.
pid_text = df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40]))
topic_inputs = [
    build_input(
        t,
        weights_df=weights_df,
        pid_text=pid_text,
        groupby_field=GROUPBY_FIELD,
        n_representative=N_REPRESENTATIVE,
        top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
    )
    for _, t in topics_df.iterrows()
]

topic_order = {
    (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
    for i, inp in enumerate(topic_inputs)
}

results = []
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _label_with_retry,
            inp,
            client=client,
            model_labeling=MODEL_LABELING,
            system_prompt=SYSTEM,
            user_prompt_template=PROMPT,
            groupby_field=GROUPBY_FIELD,
            warnings_path=WARNINGS_PATH,
            max_retries=MAX_RETRIES,
        ): inp
        for inp in topic_inputs
    }
    for future in as_completed(futures):
        inp = futures[future]
        obj = future.result()
        results.append(obj)
        print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

# Re-sort to the original topic order so downstream joins stay stable.
bad_sort_keys = []
for r in results:
    norm_key = (
        _norm_group_value(r.get(GROUPBY_FIELD, "")),
        _safe_topic_id(r.get("topic_id", -1)),
    )
    if norm_key not in topic_order:
        bad_sort_keys.append({"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")})

if bad_sort_keys:
    raise ValueError(
        f"LABELING_SORT_KEY_MISMATCH: {len(bad_sort_keys)} label result(s) did not match the input topic order. "
        f"Examples: {bad_sort_keys[:5]}"
    )

results = sorted(
    results,
    key=lambda r: topic_order[
        (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
    ],
)

with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{len(results)} labels saved")

  __agriculture_food_systems_careers__ / topic 18 → Social-emotional support for rural farm gardens
  __agriculture_food_systems_careers__ / topic 13 → Applied problem solving STEM projects
  __agriculture_food_systems_careers__ / topic 0 → Hydroponic and compost gardening instruction
  __agriculture_food_systems_careers__ / topic 4 → Dramatic farmer market role-play
  __agriculture_food_systems_careers__ / topic 9 → Chicken raising for science exploration
  __agriculture_food_systems_careers__ / topic 12 → Hydroponic and garden nutrition learning
  __agriculture_food_systems_careers__ / topic 6 → chick incubation and life-cycle observations
  __agriculture_food_systems_careers__ / topic 1 → Food systems sustainability interdisciplinary course
  __agriculture_food_systems_careers__ / topic 15 → Interdisciplinary food systems coursework
  __agriculture_food_systems_careers__ / topic 7 → Animal science library books
  __agriculture_food_systems_careers__ / topic 17 → Hydroponic fine moto

In [10]:
# ── Label post-processing ──────────────────────────────────────────────────
# Keep only parseable labeling results and validate them against source topics.

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

# Source-of-truth allowed groups come from the source topic table, not model output.
ALLOWED_GROUP_VALUES = [
    str(g)
    for g in topics_df[GROUPBY_FIELD].dropna().drop_duplicates().tolist()
]

invalid_groups = sorted(
    set(labels_df[GROUPBY_FIELD].astype(str).unique()) - set(ALLOWED_GROUP_VALUES)
)
if invalid_groups:
    raise ValueError(
        f"Invalid {GROUPBY_FIELD} values returned during labeling: {invalid_groups}"
    )

# Normalize types after validation so later joins are stable.
labels_df[GROUPBY_FIELD] = labels_df[GROUPBY_FIELD].astype(str)
labels_df["topic_id"] = labels_df["topic_id"].astype(int)

labeled_groups = set(labels_df[GROUPBY_FIELD].unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(set(ALLOWED_GROUP_VALUES) - labeled_groups)

labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

,industry,topic_id,proposed_label,coherence_flag,description
0,__agriculture_food_systems_careers__,0,Hydroponic and compost gardening instruction,coherent,Requests for hydroponic garden and composting ...
1,__agriculture_food_systems_careers__,1,Food systems sustainability interdisciplinary ...,coherent,Requests to teach food systems sustainability ...
2,__agriculture_food_systems_careers__,2,Farm animal play and counting,coherent,Requests for farm animal themed toys and math ...
3,__agriculture_food_systems_careers__,3,Hydroponic garden plant growing,coherent,Requests to build and use an indoor hydroponic...
4,__agriculture_food_systems_careers__,4,Dramatic farmer market role-play,coherent,Requests to set up dramatic play market and sh...
...,...,...,...,...,...
155,__transportation_logistics_aviation__,15,Elementary community service events,unclear,Requests for community service event support a...
156,__transportation_logistics_aviation__,16,STEM engineering design problem-solving,coherent,Requests for hands-on engineering and problem-...
157,__transportation_logistics_aviation__,17,Winter bus transportation coats support,coherent,Requests for warm winter coats and reliable sc...
158,__transportation_logistics_aviation__,18,Winter clothing for students,coherent,"Requests for distributing winter coats, gloves..."


---
## Step 6 — Synthesis

Produces two synthesis passes in sequence:

1. **Cross-group synthesis** — a single LLM call over all topic lines to identify
   patterns, contrasts, framing logic, and notable boundaries across the corpus.
2. **Per-group synthesis** — one LLM call per group, run in parallel, to surface
   each group's dominant internal mechanisms.

Both passes produce plain text saved to `analysis/llm_synthesis_*.txt` and are
assembled in Step 7 as the evidence base for the structured JSON insight call.

In [11]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
# First synthesize the whole landscape, then synthesize each group in parallel.

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "Your job is to synthesize topic-level evidence into stable, decision-useful analytic findings "
    "for internal strategy work. You are not writing a polished memo and you are not doing creative interpretation. "
    "You are performing disciplined evidence grouping.\n\n"

    "Core objective:\n"
    "Produce findings that are specific, repeatable, and tightly grounded in the supplied topic lines. "
    "Literal evidence discipline is more important than elegance.\n\n"

    "Evidence rules:\n"
    "- Treat the supplied topic lines as the full evidence base.\n"
    "- Every finding must be supported by explicitly named topics.\n"
    "- Use only topics that directly support the claim.\n"
    "- Do not generalize beyond the named supporting topics.\n"
    "- If evidence is borderline, omit the finding rather than stretching it.\n\n"

    "Merge rules:\n"
    "- Merge topics only when they share the same dominant mechanism or classroom function.\n"
    "- Do not merge topics just because they involve the same product category, school setting, or broad theme.\n"
    "- Keep access/accommodation, regulation/behavior, maintenance/operations, identity/belonging, and routine/workflow "
    "separate unless the evidence clearly shows they are the same pattern.\n"
    "- If one candidate finding is broader and another is a more specific mechanism-level version, prefer the more specific one.\n\n"

    "Stability rules:\n"
    "- Review topics in the order given.\n"
    "- Prefer narrower mechanism-level patterns over broad umbrella summaries.\n"
    "- Use the same output structure every time.\n"
    "- If two formulations are both defensible, choose the more literal and reusable one.\n\n"

    "Support-citation rules:\n"
    "- For every finding, include a 'Supporting topics' line.\n"
    "- Each supporting topic must be written exactly as <group_value>|<topic_id>.\n"
    "- group_value must be copied verbatim from the topic lines provided; do not abbreviate, paraphrase, normalize spelling, or invent names.\n"
    "- Do not write labels or prose in place of the support references.\n\n"

    "Writing rules:\n"
    "- Be precise, concrete, and analytical.\n"
    "- Do not mention NMF, model, cluster, pipeline, or analytical machinery.\n"
    "- Do not write donor-facing rhetoric.\n"
    "- Do not use vague abstractions when the evidence supports a concrete mechanism.\n"
    "- Do not invent a cleaner pattern than the evidence supports."
)

topic_lines = build_topic_lines(
    labels_df,
    GROUPBY_FIELD,
    top_terms_count=SYNTHESIS_TOP_TERMS_COUNT,
)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of topic lines discovered from teacher project request essays on DonorsChoose,
grouped by "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

Each line contains:
- group value
- topic number
- topic label
- coherence flag
- top terms
- one-sentence description

Topic lines:
{topic_lines}

Your task:
Synthesize the topic lines into a stable cross-group analysis that will later be used to generate
external-facing insights. Preserve nuance, but be disciplined about grouping.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify candidate cross-group patterns only when multiple topics share the same dominant mechanism,
   classroom function, or hidden operating logic.
3. Keep distinct mechanisms separate even when they live in similar product categories.
4. Rank findings by:
   a. breadth across groups
   b. specificity of the shared mechanism
   c. clarity of distinction from nearby themes
5. Omit weak or borderline patterns rather than padding.

Return plain text only, using exactly this structure and these section headings:

CROSS-GROUP PATTERNS
Finding 1
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: ...

IMPORTANT CONTRASTS
Finding 1
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

Finding 2
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

FRAMING AND PITCH STRATEGIES
Finding 1
Title: ...
Pattern: ...
Why it matters for interpretation: ...
Supporting topics: ...

Finding 2
Title: ...
Pattern: ...
Why it matters for interpretation: ...
Supporting topics: ...

NOTABLE ABSENCES OR BOUNDARIES
Finding 1
Title: ...
Boundary: ...
Why this boundary matters: ...
Supporting topics: ...

Hard requirements:
- CROSS-GROUP PATTERNS: return {CROSS_MIN_INSIGHTS} to {CROSS_MAX_INSIGHTS} findings
- IMPORTANT CONTRASTS: return 2 to 4 findings
- FRAMING AND PITCH STRATEGIES: return 1 to 3 findings
- NOTABLE ABSENCES OR BOUNDARIES: return 0 to 2 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same pattern in multiple sections
- Do not merge findings just because they sound similar at a high level
- Prefer mechanism-level explanations over broad summaries like “access,” “engagement,” or “support”
- If a pattern is supported by only one group, it does not belong in CROSS-GROUP PATTERNS
- If a distinction is real but narrow, put it in IMPORTANT CONTRASTS rather than inflating it into a cross-group signature

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task:
Identify the strongest, most decision-useful patterns within this single group.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify the dominant subthemes in the group.
3. Separate nearby themes unless they clearly share the same dominant mechanism or classroom function.
4. Prefer narrower mechanism-level findings over broad category summaries.
5. Omit weak or redundant findings rather than padding.

Return plain text only, using exactly this structure and these section headings:

CORE SUBTHEMES
Finding 1
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: ...

IMPORTANT INTERNAL SPLITS
Finding 1
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

Finding 2
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

FRAMING AND PITCH LOGIC
Finding 1
Title: ...
Pattern: ...
Why it matters for interpretation: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Boundary: ...
Why this boundary matters: ...
Supporting topics: ...

Hard requirements:
- CORE SUBTHEMES: return 3 to 6 findings
- IMPORTANT INTERNAL SPLITS: return 1 to 3 findings
- FRAMING AND PITCH LOGIC: return 1 to 3 findings
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 2 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same idea across multiple sections
- Use 'mixed' topics carefully; do not let one mixed topic dominate a whole finding unless the collision itself is the finding
- If one theme is primary and another is secondary, keep the finding centered on the primary theme
- Do not give generic summaries of the group label; surface mechanism-level patterns inside the group

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

synthesis_cross = _call_with_retry(
    SYNTHESIS_PROMPT_CROSS_GROUP,
    client=client,
    model_name=MODEL_SYNTHESIS,
    system_prompt=SYNTHESIS_SYSTEM,
    max_retries=MAX_RETRIES,
)
if synthesis_cross is None:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed",
        context={},
    )
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

# Excluded groups were removed from df in the Parameters cell; they cannot
# appear in ALLOWED_GROUP_VALUES. Filter only for label coverage.
groups = [
    g for g in ALLOWED_GROUP_VALUES
    if g in set(labels_df[GROUPBY_FIELD].unique())
]

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            synthesize_one_group,
            g,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            group_description=GROUP_DESCRIPTION,
            per_group_instructions=PER_GROUP_INSTRUCTIONS,
            client=client,
            model_name=MODEL_SYNTHESIS,
            system_prompt=SYNTHESIS_SYSTEM,
            warnings_path=WARNINGS_PATH,
            outpath_func=OUT,
            max_retries=MAX_RETRIES,
        ): g
        for g in groups
    }
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

# Preserve the original group order before handing results downstream.
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")

Done: __public_safety_civic_careers__
Done: __health_careers__
Done: __media_creative_industries__
Done: __robotics_engineering_careers__
Done: __green_jobs_sustainability_careers__
Done: __business_entrepreneurship_careers__
Done: __transportation_logistics_aviation__
Done: __agriculture_food_systems_careers__


---
## Step 7 — Structured Insight Extraction

Assembles the cross-group and per-group synthesis texts and asks the model to
return a structured JSON object with `key_insights` (cross-group) and `by_group`
(per-group, one list per group value). The response is validated, normalized,
and written to `insights_candidates.json` as a pre-verification snapshot.

In [12]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────
# Ask the model for a single JSON object, then normalize and validate it.

OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You turn structured internal analysis into clear, decision-facing insights for external audiences: "
    "foundations, corporate partners, policymakers, executives, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences.\n\n"

    "You are working from a structured synthesis that already groups evidence into findings, contrasts, "
    "framing logic, and boundaries, and that includes explicit Supporting topics lines. "
    "Treat those synthesized findings as the evidence base.\n\n"

    "Core objective:\n"
    "- Surface the few patterns that would genuinely change how a thoughtful outsider interprets teacher demand.\n"
    "- Prefer mechanism-level reveals over broad thematic summaries.\n"
    "- Preserve evidence discipline: every insight must be grounded in the strongest matching supporting topics.\n"
    "- Write for intelligent outsiders who are not close to classroom realities.\n\n"

    "Evidence rules:\n"
    "- Use the structured synthesis as the primary evidence source.\n"
    "- Prefer source_topics that are explicitly named in Supporting topics lines.\n"
    "- Do not infer broad support if the synthesis only gives narrow support.\n"
    "- Prefer fewer well-matched source_topics over padding with weak ones.\n"
    "- Do not merge adjacent synthesized findings unless they clearly support the same external-facing insight.\n\n"

    "Key-insight distinctness rules:\n"
    "- Key insights must be mutually distinct.\n"
    "- Two key insights are not distinct if they lead to the same outsider takeaway, the same hidden mechanism, "
    "or the same funding/strategy implication.\n"
    "- If two candidate key insights overlap materially, keep the stronger one or merge them into one sharper insight.\n"
    "- Prefer the smallest non-overlapping set of key insights that fully captures the strongest cross-group patterns.\n"
    "- Do not split one broad pattern into multiple final insights unless each split produces a meaningfully different "
    "outsider interpretation or different action implication.\n\n"

    "Selection tie-break rules:\n"
    "- When choosing between two plausible candidate insights, prefer the one with:\n"
    "  1. broader support across synthesized findings,\n"
    "  2. clearer mechanism-level distinctness,\n"
    "  3. stronger outsider takeaway,\n"
    "  4. fewer but stronger source_topics,\n"
    "  5. less overlap with already selected insights.\n\n"

    "By-group rules:\n"
    "- For each group, first identify the dominant internal mechanisms.\n"
    "- Only add another by-group insight if it is not a subcase or restatement of one already selected.\n"
    "- Prefer coverage of the group's main internal story over exhaustive enumeration.\n\n"

    "Audience and style rules:\n"
    "- Write for intelligent outsiders who are not close to classroom realities.\n"
    "- The insight should feel like a reveal, not like a category recap.\n"
    "- Focus on what teachers are actually doing, compensating for, or signaling through requests.\n"
    "- Be direct, concrete, and human.\n"
    "- Not academic.\n"
    "- Not corporate boilerplate.\n"
    "- Not donor-marketing fluff.\n"
    "- No mention of topics, NMF, models, clusters, prompts, or analytical machinery.\n"
    "- No hedging such as 'may suggest' or 'could indicate'.\n"
    "- No filler.\n\n"

    "Output rules:\n"
    "- Return ONLY valid JSON.\n"
    "- Use exactly the required schema and fields.\n"
    "- Do not add extra fields.\n"
    "- Do not include markdown fences, commentary, or explanation outside the JSON object."
)

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)

for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")

if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")

example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHTS_PROMPT = f"""
You are given a structured synthesis of classroom project request patterns.

The synthesis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

It contains:
- cross-group findings
- within-group findings
- important contrasts
- framing and pitch logic
- boundaries or non-central signals
- explicit Supporting topics lines in the format <group_value>|<topic_id>

Structured synthesis:
{synthesis_input}

Your task:
Produce a decision-facing insights document for external audiences —
foundation leaders, corporate partners, policymakers, executives, and major donors —
who are intelligent but not close to classroom realities.

The best insights should make a reader think:
"I would not have seen that from surface-level demand, and this changes how I interpret what classrooms need."

WORKING METHOD:
1. Read the structured synthesis as a set of evidence units, not just as prose.
2. Identify all plausible candidate cross-group insights.
3. Remove or merge any candidate whose core implication materially overlaps another candidate.
4. Keep only the strongest non-overlapping set.
5. Then generate the final JSON.

SELECTION CRITERIA:
- Prioritize insights that reveal hidden operating pressures, misread demand, underestimated classroom functions,
  invisible tradeoffs, or shifts in how teachers are compensating for unmet needs.
- Prefer insights that would matter for funding strategy, partnership design, policy interpretation,
  or external messaging.
- Avoid observations that merely restate what the group label already implies.
- Avoid methodological commentary and data-practitioner-only insights.
- Prefer mechanism-level reveals over broad summary language.

KEY-INSIGHT DISTINCTNESS RULE:
- Each key insight must have a distinct core implication.
- Before finalizing "key_insights", check whether any two candidate key insights share the same hidden mechanism,
  same outsider takeaway, or same funding/strategy implication.
- If they do, keep the stronger one or merge them into one sharper insight.
- Do not return multiple key insights that are different phrasings or slight extensions of the same underlying point.

MERGE RULE:
- Combine synthesized findings into one final insight only when they clearly support the same external-facing interpretation of teacher demand.
- Shared product domain, broad similarity, or adjacent classroom context is not enough by itself.
- If a broader candidate and a narrower, more decision-useful candidate are both available, prefer the narrower one.
- Do not split one broad pattern into multiple final insights unless each split changes what an outsider should conclude.

COUNT + COVERAGE REQUIREMENTS:
- Return between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} cross-group insights in "key_insights".
- Treat {CROSS_MIN_INSIGHTS} as the default target.
- Start by asking whether the strongest {CROSS_MIN_INSIGHTS} key insights already capture the cross-group story.
- Only add more than {CROSS_MIN_INSIGHTS} if an additional insight is clearly distinct, non-overlapping, strongly supported,
  and would materially change what an outside reader learns.
- Never add an insight just because room remains before {CROSS_MAX_INSIGHTS}.
- Prefer the smallest non-overlapping set within the allowed range that fully captures the strongest cross-group patterns.

- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}

- Return between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} strongest defensible insights for each group value above.
- Treat {PER_GROUP_MIN_INSIGHTS} as the default target for each group.
- Start by asking whether the strongest {PER_GROUP_MIN_INSIGHTS} insights already capture the group's main internal story.
- Only add more than {PER_GROUP_MIN_INSIGHTS} when the additional insight introduces a clearly different mechanism,
  not just a narrower example, subcase, or rephrasing.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.
- Keep by-group insights focused on the group's dominant internal mechanisms and avoid repeating the same implication in multiple phrasings.

SELECTION TIE-BREAK RULES:
- When choosing between two plausible candidate insights, prefer the one with:
  1. broader support across synthesized findings,
  2. clearer mechanism-level distinctness,
  3. stronger outsider takeaway,
  4. fewer but stronger source_topics,
  5. less overlap with already selected insights.

INSIGHT STRUCTURE:
For every insight, use exactly these fields:

- title:
  a crisp declarative headline that feels like a reveal, not a topic label

- what_seeing:
  2-4 sentences in plain, concrete language describing what teachers are actually doing, asking for,
  compensating for, or signaling through requests.
  Write as an outsider-facing interpretation of the evidence, not as a summary of an analysis process.

- why_easy_to_miss:
  1-2 sentences explaining why this pattern is hard to see from conventional reporting,
  surface-level category labels, or simplistic interpretations of teacher demand.

- source_topics:
  a list of the strongest grounding topics for the insight.
  Use only topics that directly support the claim.
  Usually 2-4 topics is enough, but do not force a fixed count.
  Prefer fewer strong matches over weak padding.
  Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the synthesis
    - topic_id must be the integer topic number for that group
  Example for this run: {example_source_topics_json}

SOURCE_TOPIC RULES:
- Prefer source_topics that are explicitly named in Supporting topics lines.
- If a synthesized finding names multiple supporting topics but only some directly support your final claim,
  include only the strongest matches.
- If a topic is illustrative but not necessary to support the claim, leave it out.
- Do not use labels, prose, section names, or invented group names in place of group_value.
- source_topics must be a list of strings only, not objects.
- Do not include any extra fields beyond: title, what_seeing, why_easy_to_miss, source_topics.

STYLE:
- Direct, confident, and human.
- Not academic.
- Not corporate boilerplate.
- Not donor-marketing fluff.
- No mention of topics, models, clusters, or analytical machinery.
- No hedging phrases like "may suggest" or "could indicate".
- No filler.

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.
- Every source_topics item must follow the exact required format.
- Every insight object must contain exactly these four fields:
  title, what_seeing, why_easy_to_miss, source_topics

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* {CROSS_MIN_INSIGHTS}-{CROSS_MAX_INSIGHTS} insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* {PER_GROUP_MIN_INSIGHTS}-{PER_GROUP_MAX_INSIGHTS} insight objects */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user", "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = strip_json_fences(resp.choices[0].message.content)
insights_data = json.loads(raw_json)

if not isinstance(insights_data, dict):
    raise ValueError("Top-level insights response is not a JSON object.")

raw_key = insights_data.get("key_insights", [])
raw_by_group = insights_data.get(OUTPUT_GROUP_KEY, {})

if not isinstance(raw_key, list):
    raise ValueError("key_insights is not a list.")
if not isinstance(raw_by_group, dict):
    raise ValueError(f"{OUTPUT_GROUP_KEY} is not an object.")

normalized = {
    "key_insights": [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in raw_key
    ],
    OUTPUT_GROUP_KEY: {},
}

missing_group_values = [g for g in REQUIRED_GROUP_VALUES if g not in raw_by_group]
if missing_group_values:
    raise ValueError(f"Model omitted required group values: {missing_group_values}")

extra_group_values = [g for g in raw_by_group.keys() if g not in REQUIRED_GROUP_VALUES]
if extra_group_values:
    print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

for group_value in REQUIRED_GROUP_VALUES:
    items = raw_by_group.get(group_value, [])
    if not isinstance(items, list):
        raise ValueError(f"{OUTPUT_GROUP_KEY}['{group_value}'] is not a list.")
    normalized[OUTPUT_GROUP_KEY][group_value] = [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in items
    ]

insights_data = normalized

if not (CROSS_MIN_INSIGHTS <= len(insights_data["key_insights"]) <= CROSS_MAX_INSIGHTS):
    raise ValueError(
        f"Expected between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} key insights, "
        f"got {len(insights_data['key_insights'])}"
    )

bad_group_counts = {
    group_value: len(items)
    for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    if not (PER_GROUP_MIN_INSIGHTS <= len(items) <= PER_GROUP_MAX_INSIGHTS)
}
if bad_group_counts:
    raise ValueError(
        f"Expected between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} insights per group value, "
        f"got: {bad_group_counts}"
    )

insights_candidates_for_save = {
    "key_insights": [
        project_insight_for_saved_candidates(i)
        for i in insights_data["key_insights"]
    ],
    OUTPUT_GROUP_KEY: {
        group_value: [
            project_insight_for_saved_candidates(i)
            for i in items
        ]
        for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    },
}

# Mid-cell write required by spec: post-normalize_insight(), pre-verification.
write_json(OUT("insights", "insights_candidates.json"), insights_candidates_for_save)

PosixPath('/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/industry/2026-05-14/20260514_112716_industry_0c79fb23/insights/insights_candidates.json')

---
## Step 8 — Topic Verification

Verifies that each claimed source topic genuinely supports its insight. Topics
that are too broad, indirect, or adjacent are dropped.

Verification strategy:
- **Cross-group insights** — always verified.
- **By-group insights** — verified only when they cite ≥ `by_group_min_source_topics`
  source topics; narrow local findings with 1–2 cited topics pass through without
  an additional API call.

In [13]:
# ── Topic verification ─────────────────────────────────────────────────────
# Verify source-topic grounding for final insights.
# Strategy:
# - always verify key_insights
# - verify by-group insights only when they cite 3+ source topics
#   (broad claims are more likely to overclaim support than narrow local ones)

VERIFY_SYSTEM = (
    "You are validating whether cited topics directly support an insight. "
    "Be strict. Keep a topic only if its label and description directly support the claim. "
    "If support is broad, indirect, adjacent, or only loosely related, drop it. "
    "Prefer false negatives to false positives. "
    "Return only valid JSON."
)

VERIFY_BY_GROUP_MIN_SOURCE_TOPICS = VERIFY_CFG["by_group_min_source_topics"]

# Always verify cross-group/key insights
insights_data["key_insights"], key_verify_stats = _verify_insight_list(
    insights_data.get("key_insights", []),
    labels_df=labels_df,
    groupby_field=GROUPBY_FIELD,
    required_group_values=REQUIRED_GROUP_VALUES,
    client=client,
    model_verify=MODEL_VERIFY,
    system_prompt=VERIFY_SYSTEM,
    warnings_path=WARNINGS_PATH,
    min_source_topics_to_verify=1,
)

group_verify_stats = {}
for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    verified_items, stats = _verify_insight_list(
        insights_data[OUTPUT_GROUP_KEY][cat],
        labels_df=labels_df,
        groupby_field=GROUPBY_FIELD,
        required_group_values=REQUIRED_GROUP_VALUES,
        client=client,
        model_verify=MODEL_VERIFY,
        system_prompt=VERIFY_SYSTEM,
        warnings_path=WARNINGS_PATH,
        min_source_topics_to_verify=VERIFY_BY_GROUP_MIN_SOURCE_TOPICS,
    )
    insights_data[OUTPUT_GROUP_KEY][cat] = verified_items
    group_verify_stats[cat] = stats

total_group_changed = sum(v["changed_count"] for v in group_verify_stats.values())
total_group_zeroed = sum(v["dropped_to_zero_count"] for v in group_verify_stats.values())
total_group_topics_before = sum(v["topics_before"] for v in group_verify_stats.values())
total_group_topics_after = sum(v["topics_after"] for v in group_verify_stats.values())

print("Verification complete.")
print(
    f"Key insights: {key_verify_stats['insight_count']} insights | "
    f"{key_verify_stats['changed_count']} changed | "
    f"{key_verify_stats['dropped_to_zero_count']} dropped to zero topics | "
    f"{key_verify_stats['topics_before']} -> {key_verify_stats['topics_after']} source topics"
)
print(
    f"By-group insights: {sum(v['insight_count'] for v in group_verify_stats.values())} insights | "
    f"{total_group_changed} changed | "
    f"{total_group_zeroed} dropped to zero topics | "
    f"{total_group_topics_before} -> {total_group_topics_after} source topics | "
    f"verified only when source_topics >= {VERIFY_BY_GROUP_MIN_SOURCE_TOPICS}"
)

Adjusted source_topics: Hands-on living systems are being used as daily science infrastructure, not just | 3 -> 2
Adjusted source_topics: This group contains one of the clearest true career-pathway signals: students us | 3 -> 2
Adjusted source_topics: Business demand splits cleanly between learning money and running a venture | 4 -> 0
Adjusted source_topics: Regulation support is a major classroom function here, and it is more specialize | 4 -> 3
Adjusted source_topics: The true career-pathway strand in this group shows up when students simulate med | 3 -> 2
Adjusted source_topics: The important split in media requests is between making media and merely accessi | 4 -> 3
Adjusted source_topics: Career-readiness language becomes real when students are operating production ge | 3 -> 2
Adjusted source_topics: Not all expressive materials in this group are vocational; some are being used t | 3 -> 1
Adjusted source_topics: Robots are often the entry point to coding, not the end goal | 3 -> 1

---
## Step 9 — Evidence Tables

Expands each verified insight into a flat summary row (`insights_flat.csv`) and
a per-topic support detail row (`insight_topic_support.csv`). Projects are ranked
by combined topic_share across all of an insight's verified topics so Looker
links surface the most representative essays.

In [14]:
# ── VERIFIED INSIGHT SUPPORT TABLES ────────────────────────────────────────

assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing — ensure the bridge-build cell ran successfully"
)

BRIDGE_LOOKUP = build_bridge_lookup(project_topic_bridge_df)

label_index = build_label_index(
    labels_df,
    groupby_field=GROUPBY_FIELD,
    warnings_path=WARNINGS_PATH,
)

insights_flat_df, insight_topic_support_df = build_verified_insight_tables(
    insights_data,
    OUTPUT_GROUP_KEY,
    groupby_field=GROUPBY_FIELD,
    bridge_lookup=BRIDGE_LOOKUP,
    label_index=label_index,
    run_id=RUN_ID,
    # csv_max_ids_per_insight controls insight row density; see params.yaml.
    top_project_id_limit=CSV_MAX_IDS_PER_INSIGHT,
)

insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)
insight_topic_support_df.to_csv(
    OUT("insights", "insight_topic_support_candidates.csv"),
    index=False,
)

print(f"Candidate insights summarized: {len(insights_flat_df):,}")

Candidate insights summarized: 28


---
## Step 10 — Packaging, Dedupe & Topline Selection

Three sequential operations:

1. **Packaging** — apply quality threshold gates. Insights that don't clear
   `min_verified_topic_count`, `min_supporting_project_count`, and
   `min_mean_topic_share` are rejected.
2. **Deduplication** — deterministic overlap-based dedup within pair types.
3. **Topline selection** — assigns `report_section` to each accepted insight:
   `main_cross` (top N by quality rank), `main_by_group` (top 1 per group),
   `appendix_cross`, `appendix_by_group`.

In [15]:
# ── PACKAGING + DEDUPE + SIMPLE TOPLINE CUT ────────────────────────────────
# Business rule:
# - accept insights that clear threshold gates
# - remove obvious duplicates
# - then choose topline from the remaining pool
#
# Main section:
# - top N cross-category insights by:
#     1) supporting_project_count
#     2) verified_topic_count
#     3) mean_topic_share_all_verified_topics
# - top 1 by-group insight per category using the same ranking
#
# Appendix:
# - all other accepted insights

packaging = apply_deterministic_packaging(
    insights_flat_df,
    output_group_key=OUTPUT_GROUP_KEY,
    packaging_cfg=PACKAGING_CFG,
)

accepted_pack_df = packaging["accepted_df"]

print(f"Total insights before packaging: {len(insights_flat_df)}")
print(f"Accepted after thresholds: {len(accepted_pack_df)}")

curated_df, dedup_audit_df = dedupe_packaged_insights(
    accepted_pack_df,
    dedupe_cfg=DEDUPE_CFG,
)

curated_df = curated_df.copy()
curated_df["looker_url"] = curated_df["top_project_ids"].apply(
    lambda ids: build_looker_project_url(
        base_url=LOOKER_BASE_URL,
        project_ids=ids,
        filter_field=LOOKER_FILTER_FIELD,
        fields=LOOKER_FIELDS,
        limit=LOOKER_LIMIT,
        max_ids=LOOKER_ID_LIMIT,
    )
)

# main_min_verification_ratio gates eligibility for main-section placement;
# sourced from params.yaml → analysis.packaging.main_min_verification_ratio.
curated_df = assign_topline_sections_simple(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
    main_cross_limit=PACKAGING_CFG["main_cross_limit"],
    main_min_verification_ratio=MAIN_MIN_VERIFICATION_RATIO,
)

main_cross_df = curated_df[curated_df["report_section"] == "main_cross"].copy()
main_by_group_df = curated_df[curated_df["report_section"] == "main_by_group"].copy()
appendix_df = curated_df[
    curated_df["report_section"].isin(["appendix_cross", "appendix_by_group"])
].copy()

accepted_ids = set(curated_df["insight_id"])
rejected_ids = set(insights_flat_df["insight_id"]) - accepted_ids

# export project_id list by insight
insight_project_df = (
    insight_topic_support_df[
        insight_topic_support_df["insight_id"].isin(accepted_ids)
    ][["insight_id", "group_value", "topic_id"]]
    .merge(
        project_topic_bridge_df[["project_id", GROUPBY_FIELD, "topic_id"]],
        left_on=["group_value", "topic_id"],
        right_on=[GROUPBY_FIELD, "topic_id"],
        how="left",
    )
    [["insight_id", "project_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
insight_project_df.to_csv(OUT("insights", "insight_project_bridge.csv"), index=False)
curated_df.to_csv(OUT('chart_data', 'curated_df.csv'), index=False)

print(f"Accepted before dedupe: {len(accepted_pack_df)}")
print(f"Dropped as obvious duplicates: {len(dedup_audit_df)}")
print(f"Accepted after dedupe: {len(curated_df)}")
print(f"Main cross-category selected: {len(main_cross_df)}")
print(f"Main by-group selected: {len(main_by_group_df)}")
print(f"Appendix selected: {len(appendix_df)}")
print(f"{len(insight_project_df):,} insight-project pairs across {insight_project_df['insight_id'].nunique()} insights")

Total insights before packaging: 28
Accepted after thresholds: 27
Accepted before dedupe: 27
Dropped as obvious duplicates: 0
Accepted after dedupe: 27
Main cross-category selected: 3
Main by-group selected: 8
Appendix selected: 16
26,012 insight-project pairs across 27 insights


---
## Step 11 — Final Outputs & Manifests

Persists all accepted and rejected outputs, builds the DOCX report, and writes
stage and pipeline manifests. Running this cell again after a partial run is
safe — all writes are deterministic.

In [16]:
# ── FINAL OUTPUTS + REPORTING ──────────────────────────────────────────────
# Persist accepted/rejected outputs, evidence exports, chart data, DOCX, and manifests.

structured = build_structured_from_curated(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
)

write_json(OUT("insights", "insights_structured.json"), structured)

rejected_df = insights_flat_df[
    ~insights_flat_df["insight_id"].isin(curated_df["insight_id"])
].copy()
write_json(
    OUT("insights", "rejected_insights.json"),
    rejected_df.to_dict(orient="records"),
)

insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].to_csv(
    OUT("insights", "insight_topic_support.csv"),
    index=False,
)

chart_ready_insights_df = curated_df[
    [
        "run_id",
        "insight_id",
        "title",
        "section",
        "category_bucket",
        "supporting_project_count",
        "verified_topic_count",
        "mean_topic_share_all_verified_topics",
        "verification_ratio",
        "report_section",
        "report_order",
    ]
].rename(columns={"title": "theme"})

chart_ready_group_support_df = insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].copy()

chart_ready_insights_df.to_csv(
    OUT("chart_data", "chart_ready_insights.csv"),
    index=False,
)
chart_ready_group_support_df.to_csv(
    OUT("chart_data", "chart_ready_group_support.csv"),
    index=False,
)

docx_path = OUT("reports", "trend_tracker_report.docx")
build_packaged_report_docx(
    structured=structured,
    output_path=docx_path,
    output_group_key=OUTPUT_GROUP_KEY,
    report_cfg=CFG["output"],
    project_count=len(df),
    run_id=RUN_ID,
)

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())
manifest_status = "failure" if groups_failed else "success"

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status=manifest_status,
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/06_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(COPIED_CONFIG_PATH, "resolved_params_yaml"),
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insight_topic_support.csv"), "insight_topic_support_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_group_support.csv"), "chart_ready_group_support_csv"),
        artifact_meta(OUT("reports", "trend_tracker_report.docx"), "trend_tracker_report_docx"),
    ],
    row_counts={
        "input_projects": int(len(raw_df)),
        "filtered_projects": int(len(df)),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(insights_flat_df)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
    },
    key_params={
        "group_by": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "min_group_projects": CFG["analysis"]["min_group_projects"],
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "model_labeling": MODEL_LABELING,
        "model_synthesis": MODEL_SYNTHESIS,
        "verification_config": VERIFY_CFG,
        "packaging_config": PACKAGING_CFG,
        "dedupe_config": DEDUPE_CFG,
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    status=manifest_status,
    config_path=MANIFEST_CFG_PATH,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "rejected_insights_json": str(OUT("insights", "rejected_insights.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support_csv": str(OUT("insights", "insight_topic_support.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "chart_ready_group_support_csv": str(OUT("chart_data", "chart_ready_group_support.csv")),
        "trend_tracker_report_docx": str(OUT("reports", "trend_tracker_report.docx")),
    },
)

print(f"Accepted insights: {len(accepted_ids):,}")
print(f"Rejected insights: {len(rejected_ids):,}")
print(f"Docx saved → {docx_path}")
print(f"Stage manifest written → {stage_manifest_path}")
print(f"Pipeline manifest written → {pipeline_manifest_path}")

Accepted insights: 27
Rejected insights: 1
Docx saved → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/industry/2026-05-14/20260514_112716_industry_0c79fb23/reports/trend_tracker_report.docx
Stage manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/industry/2026-05-14/20260514_112716_industry_0c79fb23/metadata/stage_manifest_03_insights_generation.json
Pipeline manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/industry/2026-05-14/20260514_112716_industry_0c79fb23/metadata/pipeline_manifest.json
